# Khởi tạo và Đẩy dữ liệu vào PostgreSQL

Notebook này chứa quy trình ingest dữ liệu bao gồm đọc file `feature_records.json` chứa đặc trưng âm thanh, parse metadata từ thư mục `raw` và đẩy toàn bộ vào CSDL PostgreSQL qua pgvector.

In [7]:
import json
import psycopg2
import os

# Cấu hình kết nối DB (chuẩn theo docker-compose.yml)
DB_CONFIG = {
    'dbname': 'voice_db',
    'user': 'admin',
    'password': 'admin_password',
    'host': 'localhost',
    'port': '5432'
}

# Các đường dẫn cần thiết (tương đối từ notebook này ra ngoài data)
DATA_BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'data'))
SPEAKER_INFO_PATH = os.path.join(DATA_BASE_DIR, 'raw', 'VCTK-Corpus', 'VCTK-Corpus', 'speaker-info.txt')
FEATURES_JSON_PATH = os.path.join(DATA_BASE_DIR, 'feature_records.json')
PROCESSED_DATA_DIR = os.path.join(DATA_BASE_DIR, 'processed')

### Định nghĩa hàm tải siêu dữ liệu Diễn giả

In [8]:
def load_speaker_info():
    """Hàm đọc thông tin diễn giả (Giới tính, tuổi, accent) từ speaker-info.txt"""
    speakers = {}
    if not os.path.exists(SPEAKER_INFO_PATH):
        print(f"Cảnh báo: Không tìm thấy file speaker-info tại {SPEAKER_INFO_PATH}")
        return speakers
    
    with open(SPEAKER_INFO_PATH, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines[1:]: # Bỏ qua header
            parts = line.strip().split()
            if len(parts) >= 4:
                spk_id = f"p{parts[0]}"
                age = parts[1]
                gender_code = parts[2]
                accent = parts[3]
                
                # Chuyển đổi giới tính theo cấu trúc CHECK in ('male', 'female') của CSDL
                gender = 'female' if gender_code.upper() == 'F' else ('male' if gender_code.upper() == 'M' else None)
                
                try:
                    age = int(age)
                except ValueError:
                    age = None
                    
                speakers[spk_id] = {
                    'age': age,
                    'gender': gender,
                    'accent': accent
                }
    return speakers

### Định nghĩa hàm đẩy dữ liệu (Ingest)

In [9]:
def ingest_data():
    """Đọc dữ liệu từ thư mục processed và feature_records.json, rồi gộp metadata đẩy lên CSDL"""
    conn = None
    try:
        # 1. Kết nối PostgreSQL
        print("Đang kết nối vào cơ sở dữ liệu PostgreSQL...")
        conn = psycopg2.connect(**DB_CONFIG)
        cur = conn.cursor()
        
        # 2. Tải metadata của các speaker
        print("Đang tải thông tin các speaker...")
        speakers_info = load_speaker_info()
        
        # 3. Mở file records đặc trưng đã được phân tích
        if not os.path.exists(FEATURES_JSON_PATH):
            print(f"Lỗi: Không có dữ liệu ở {FEATURES_JSON_PATH}. Xin thực hiện extract features trước!")
            return

        print("Đang nạp file đặc trưng âm thanh được trích xuất từ data/processed...")
        with open(FEATURES_JSON_PATH, 'r', encoding='utf-8') as f:
            records = json.load(f)
            
        print(f"Tìm thấy {len(records)} bản ghi. Đang tiến hành làm sạch trước khi đẩy vào DB...")
        
        inserted_count = 0
        for rec in records:
            file_path = rec.get('file_path')
            
            # Chỉ nạp những record mà file .wav thực sự tồn tại ở thư mục data/processed
            # Đối với jupyter root path sẽ dựa trên os.getcwd()
            wav_absolute_path = os.path.join(DATA_BASE_DIR, file_path)
            if not os.path.exists(wav_absolute_path):
                continue
                
            speaker = rec.get('speaker')
            npy_path = rec.get('npy_path')
            sample_rate = rec.get('sample_rate')
            duration_sec = rec.get('duration_sec')
            embedding = rec.get('embedding')
            
            if not embedding or len(embedding) != 99:
                continue
            
            # Khớp thông tin demographic của speaker
            spk_info = speakers_info.get(speaker, {})
            age = spk_info.get('age')
            gender = spk_info.get('gender')
            accent = spk_info.get('accent')
            
            # Format mảng Python dạng chuỗi array hợp chuẩn pgvector '[x, y, z...]'
            embedding_str = f"[{','.join(map(str, embedding))}]"
            
            # Execute câu lệnh INSERT
            cur.execute("""
                INSERT INTO voice_records
                    (speaker, accent, gender, age, file_path, npy_path,
                     duration_sec, sample_rate, embedding)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s::vector)
            """, (
                speaker, accent, gender, age, file_path, npy_path,
                duration_sec, sample_rate, embedding_str
            ))
            
            inserted_count += 1
            if inserted_count % 500 == 0:
                print(f"  + Đã đẩy {inserted_count} bản ghi...")
                
        # Commit giao dịch
        conn.commit()
        cur.close()
        print(f"Thành công! Đã đẩy tổng cộng {inserted_count} vector và metadata vào CSDL.")
        
    except psycopg2.Error as e:
        if conn:
            conn.rollback()
        print(f"Lỗi Database PostgreSQL: {e}")
    except Exception as e:
        if conn:
            conn.rollback()
        print(f"Lỗi: {e}")
    finally:
        if conn:
            conn.close()

### Chạy Script

In [10]:
ingest_data()

Đang kết nối vào cơ sở dữ liệu PostgreSQL...
Đang tải thông tin các speaker...
Đang nạp file đặc trưng âm thanh được trích xuất từ data/processed...
Tìm thấy 500 bản ghi. Đang tiến hành làm sạch trước khi đẩy vào DB...
  + Đã đẩy 500 bản ghi...
Thành công! Đã đẩy tổng cộng 500 vector và metadata vào CSDL.
